In [34]:
import pandas as pd

## Загружаем CSV файл, указывая ID как индексный столбец

In [35]:
df2 = pd.read_csv('../data/auto.csv', index_col='ID')

## Подсчет наблюдений

In [36]:
df2.count()

CarNumber       931
Make_n_model    931
Refund          914
Fines           869
History          82
dtype: int64

## Удаление дубликатов

In [37]:
df2 = df2.drop_duplicates(subset=['CarNumber', 'Make_n_model', 'Fines'], keep='last')
df2.count()

CarNumber       725
Make_n_model    725
Refund          713
Fines           665
History          65
dtype: int64

## Работа с пропущенными значениями

In [38]:
# Проверяем количество пропусков в каждом столбце
df2.isnull().sum()

CarNumber         0
Make_n_model      0
Refund           12
Fines            60
History         660
dtype: int64

In [39]:
# Удаляем столбцы с более чем 500 пропусками
thresh_value = len(df2) - 500
df2 = df2.dropna(axis=1, thresh=thresh_value)
print(df2.isnull().sum())

CarNumber        0
Make_n_model     0
Refund          12
Fines           60
dtype: int64


In [40]:
# Заменяем пропуски в столбце Refund предыдущим значением
df2['Refund'] = df2['Refund'].ffill()
print(df2['Refund'].isnull().sum())

0


In [41]:
if df2['Refund'].isnull().any():
    df2['Refund'] = df2['Refund'].bfill()

In [42]:
# Заменяем пропуски в столбце Fines средним значением
mean_fines = df2['Fines'].mean()
df2['Fines'] = df2['Fines'].fillna(mean_fines)
print(df2['Fines'].mean())
print(df2['Refund'].mean())

8594.586466165412
1.5172413793103448


In [43]:
df2.isnull().sum()

CarNumber       0
Make_n_model    0
Refund          0
Fines           0
dtype: int64

## Разделение марки и модели

In [44]:
def extract_make_model(row):
    if pd.isna(row['Make_n_model']):
        return pd.Series([None, None])
    parts = row['Make_n_model'].split(' ', 1)
    if len(parts) == 2:
        return pd.Series([parts[0], parts[1]])
    else:
        return pd.Series([parts[0], None])

df2[['Make', 'Model']] = df2.apply(extract_make_model, axis=1)
print(df2[['Make_n_model', 'Make', 'Model']].head(10))

     Make_n_model    Make    Model
ID                                
0      Ford Focus    Ford    Focus
1    Toyota Camry  Toyota    Camry
2      Ford Focus    Ford    Focus
3      Ford Focus    Ford    Focus
5      Ford Focus    Ford    Focus
10     Ford Focus    Ford    Focus
11     Ford Focus    Ford    Focus
12     Ford Focus    Ford    Focus
13  Skoda Octavia   Skoda  Octavia
14     Ford Focus    Ford    Focus


In [45]:
# Удаляем столбец Make_n_Model
df2 = df2.drop('Make_n_model', axis=1)
print(df2.head())

       CarNumber  Refund   Fines    Make  Model
ID                                             
0   Y163O8161RUS     2.0  3200.0    Ford  Focus
1    E432XX77RUS     1.0  6500.0  Toyota  Camry
2    7184TT36RUS     1.0  2100.0    Ford  Focus
3   X582HE161RUS     2.0  2000.0    Ford  Focus
5   92918M178RUS     1.0  5700.0    Ford  Focus


## Сохранение в JSON

In [46]:
df2.to_json('../data/auto.json', orient='records', indent=2)

In [47]:
df2 = pd.read_json('../data/auto.json', orient='records')

In [48]:
df2.count()

CarNumber    725
Refund       725
Fines        725
Make         725
Model        716
dtype: int64